In [1]:
# 1

import pandas as pd
import numpy as np

from pathlib import Path

import hdbscan

from sklearn.metrics import (
    roc_auc_score
)

from hdbscan.validity import (
    validity_index
)

In [2]:
# 2

# ----------------------------------------
# CHECKPOINT FILE
# ----------------------------------------

CHECKPOINT_FILE = (
    Path(
        "paper_implementation"
    )
    /
    "checkpoint.txt"
)

# ----------------------------------------
# INITIALIZE CHECKPOINT
# ----------------------------------------

if not CHECKPOINT_FILE.exists():

    with open(
        CHECKPOINT_FILE,
        "w"
    ) as f:

        f.write(
            "MODE=1\n"
        )

        f.write(
            "DAY=1\n"
        )

# ----------------------------------------
# LOAD CHECKPOINT
# ----------------------------------------

checkpoint = {}

with open(
    CHECKPOINT_FILE,
    "r"
) as f:

    for line in f:

        key, value = (
            line
            .strip()
            .split("=")
        )

        checkpoint[key] = (
            int(value)
        )

# ----------------------------------------
# CURRENT MODE
# ----------------------------------------

mode_number = (
    checkpoint["MODE"]
)

MODE = (
    f"mode_{mode_number}"
)

# ----------------------------------------
# CURRENT DAY
# ----------------------------------------

day_number = (
    checkpoint["DAY"]
)

DAY_INDEX = (
    day_number - 1
)

# ----------------------------------------
# MAP@K
# ----------------------------------------

K = 10

# ----------------------------------------
# DISPLAY CURRENT STATUS
# ----------------------------------------

print()

print(
    f"Current Mode : {MODE}"
)

print(
    f"Current Day  : {day_number}"
)

print()


Current Mode : mode_6
Current Day  : 1



In [3]:
# 3

BASE_DIR = Path(
    "paper_implementation"
)

NORMALIZED_DIR = (
    BASE_DIR /
    "preprocessing" /
    "normalized_data"
)

INPUT_DIR = (
    NORMALIZED_DIR /
    MODE
)

RESULTS_DIR = (
    BASE_DIR /
    "results"
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

MODE_RESULTS_DIR = (
    RESULTS_DIR /
    MODE
)

MODE_RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [4]:
# 4

hour_cols = [
    f"HOUR_{i}"
    for i in range(24)
]

In [5]:
# 5

fraud_file = (
    BASE_DIR /
    "area_assignments" /
    "fraud_consumers.csv"
)

fraud_df = pd.read_csv(
    fraud_file
)

fraud_consumers = (
    fraud_df["Meter"]
    .astype(str)
    .tolist()
)

print(
    "Fraud Consumers Loaded:",
    len(fraud_consumers)
)

Fraud Consumers Loaded: 27


In [6]:
# 6

def create_daily_matrix(
    input_dir,
    day_index
):

    files = sorted(
        input_dir.glob("*.csv")
    )

    matrix_rows = []

    meter_ids = []

    for file in files:

        meter_id = (
            file.stem
        )

        df = pd.read_csv(file)

        row = (
            df.loc[
                day_index,
                hour_cols
            ]
            .astype(float)
            .values
        )

        matrix_rows.append(
            row
        )

        meter_ids.append(
            meter_id
        )

    O = np.array(
        matrix_rows
    )

    return O, meter_ids

In [7]:
# 7

def calculate_mapk(
    ranked_consumers,
    fraud_consumers,
    k=10
):

    ranked_top_k = (
        ranked_consumers[:k]
    )

    precision_values = []

    relevant_count = 0

    for idx, consumer in enumerate(
        ranked_top_k,
        start=1
    ):

        if consumer in fraud_consumers:

            relevant_count += 1

            precision = (
                relevant_count / idx
            )

            precision_values.append(
                precision
            )

    if len(precision_values) == 0:

        return 0

    return np.mean(
        precision_values
    )

In [8]:
# 8

def tune_hdbscan_parameters(
    O
):

    best_score = -1

    best_m = None

    best_k = None

    for m in range(2, 21):

        for k in range(2, 21):

            try:

                clusterer = (
                    hdbscan.HDBSCAN(
                        min_cluster_size=m,
                        min_samples=k,
                        metric="euclidean"
                    )
                )

                cluster_labels = (
                    clusterer.fit_predict(O)
                )

                unique_labels = set(
                    cluster_labels
                )

                # --------------------------------
                # Skip invalid clustering
                # --------------------------------

                if len(unique_labels) <= 1:

                    continue

                dbcv_score = (
                    validity_index(
                        O,
                        cluster_labels,
                        metric="euclidean"
                    )
                )

                if np.isnan(dbcv_score):

                    continue

                if dbcv_score > best_score:

                    best_score = (
                        dbcv_score
                    )

                    best_m = m

                    best_k = k

            except:

                continue

    return (
        best_m,
        best_k,
        best_score
    )

In [9]:
# 9

def run_hdbscan(
    O,
    m,
    k
):

    clusterer = (
        hdbscan.HDBSCAN(
            min_cluster_size=m,
            min_samples=k,
            metric="euclidean"
        )
    )

    clusterer.fit(O)

    labels = (
        clusterer.labels_
    )

    outlier_scores = (
        clusterer.outlier_scores_
    )

    return (
        labels,
        outlier_scores
    )

In [10]:
# 10

def rank_consumers(
    meter_ids,
    labels,
    outlier_scores
):

    results = pd.DataFrame({

        "Meter_ID":
        meter_ids,

        "Cluster_Label":
        labels,

        "Outlier_Score":
        outlier_scores
    })

    # ----------------------------------------
    # Keep ONLY noise consumers
    # ----------------------------------------

    noise_df = (
        results[
            results[
                "Cluster_Label"
            ] == -1
        ]
        .copy()
    )

    noise_df = (
        noise_df.sort_values(
            by="Outlier_Score",
            ascending=False
        )
        .reset_index(drop=True)
    )

    ranked_consumers = (
        noise_df[
            "Meter_ID"
        ].tolist()
    )

    return (
        ranked_consumers,
        noise_df
    )

In [11]:

# 11

def calculate_paper_auc(
    all_ranked_consumers,
    fraud_consumers,
    total_consumers
):

    # ----------------------------------------
    # Paper AUC:
    # ascending order of anomaly scores
    # ----------------------------------------

    ascending_ranked = list(
        reversed(
            all_ranked_consumers
        )
    )

    fraud_ranks = []

    for idx, meter_id in enumerate(
        ascending_ranked
    ):

        if meter_id in fraud_consumers:

            fraud_ranks.append(
                idx + 1
            )

    F = len(fraud_ranks)

    U = (
        total_consumers
        -
        F
    )

    if F == 0 or U == 0:

        return 0.5

    auc_score = (
        (
            sum(fraud_ranks)
            -
            (
                0.5
                *
                F
                *
                (
                    F + 1
                )
            )
        )
        /
        (
            F
            *
            U
        )
    )

    return auc_score


def compute_metrics(
    meter_ids,
    outlier_scores,
    fraud_consumers,
    ranked_consumers,
    k=10
):

    # ----------------------------------------
    # Replace NaN outlier scores
    # ----------------------------------------

    outlier_scores = np.nan_to_num(
        outlier_scores,
        nan=0.0
    )

    # ----------------------------------------
    # y_true for sklearn ROC-AUC
    # ----------------------------------------

    y_true = []

    for meter_id in meter_ids:

        if meter_id in fraud_consumers:

            y_true.append(1)

        else:

            y_true.append(0)

    # ----------------------------------------
    # sklearn ROC-AUC
    # ----------------------------------------

    if len(np.unique(outlier_scores)) == 1:

        sklearn_auc = 0.5

    else:

        sklearn_auc = (
            roc_auc_score(
                y_true,
                outlier_scores
            )
        )

    # ----------------------------------------
    # Create ALL-consumer ranking
    # ----------------------------------------

    all_results = pd.DataFrame({

        "Meter_ID": meter_ids,

        "Outlier_Score": outlier_scores

    })

    all_results = all_results.sort_values(

        by="Outlier_Score",

        ascending=False

    )

    all_ranked_consumers = (
        all_results[
            "Meter_ID"
        ].tolist()
    )

    # ----------------------------------------
    # Paper AUC
    # ----------------------------------------

    paper_auc = (
        calculate_paper_auc(
            all_ranked_consumers,
            fraud_consumers,
            len(meter_ids)
        )
    )

    # ----------------------------------------
    # MAP@10
    # ----------------------------------------

    mapk_score = (
        calculate_mapk(
            ranked_consumers,
            fraud_consumers,
            k
        )
    )

    return (

        sklearn_auc,

        paper_auc,

        mapk_score
    )


In [12]:
#12

# ----------------------------------------
# Create Matrix O
# ----------------------------------------

O, meter_ids = (
    create_daily_matrix(
        INPUT_DIR,
        DAY_INDEX
    )
)

print(
    "Matrix O Shape:",
    O.shape
)

# ----------------------------------------
# Hyperparameter Tuning
# ----------------------------------------

best_m, best_k, best_dbcv = (
    tune_hdbscan_parameters(
        O
    )
)

# ----------------------------------------
# Safety Check
# ----------------------------------------

if best_m is None:

    raise ValueError(
        "No valid clustering found."
    )

print(
    "Best min_cluster_size:",
    best_m
)

print(
    "Best min_samples:",
    best_k
)

print(
    "Best DBCV:",
    best_dbcv
)

# ----------------------------------------
# Final HDBSCAN
# ----------------------------------------

labels, outlier_scores = (
    run_hdbscan(
        O,
        best_m,
        best_k
    )
)

# ----------------------------------------
# Ranking
# ----------------------------------------

ranked_consumers, noise_df = (
    rank_consumers(
        meter_ids,
        labels,
        outlier_scores
    )
)

print(
    "Noise Consumers:",
    len(noise_df)
)

# ----------------------------------------
# Metrics
# ----------------------------------------

sklearn_auc, paper_auc, mapk_score = (
    compute_metrics(
        meter_ids,
        outlier_scores,
        fraud_consumers,
        ranked_consumers,
        K
    )
)

print(
    "sklearn ROC-AUC:",
    sklearn_auc
)

print(
    "Paper AUC:",
    paper_auc
)

print(
    "MAP@10:",
    mapk_score
)

Matrix O Shape: (0,)


ValueError: No valid clustering found.

In [ ]:
# 13

ranked_output_file = (
    MODE_RESULTS_DIR /
    f"ranked_day_{DAY_INDEX + 1:02d}.csv"
)

noise_df.to_csv(
    ranked_output_file,
    index=False
)

print(
    "Ranked consumers saved."
)

In [ ]:
# 14

metrics_file = (
    MODE_RESULTS_DIR /
    "daily_metrics.csv"
)

current_result = pd.DataFrame({

    "Day":
    [DAY_INDEX + 1],

    "Best_m":
    [best_m],

    "Best_k":
    [best_k],

    "DBCV":
    [best_dbcv],

    "Sklearn_AUC":
    [sklearn_auc],

    "Paper_AUC":
    [paper_auc],

    "MAP@10":
    [mapk_score],

    "Noise_Consumers":
    [len(noise_df)]
})

# ----------------------------------------
# Safe append/update
# ----------------------------------------

if metrics_file.exists():

    old_df = pd.read_csv(
        metrics_file
    )

    # Remove existing same-day row

    old_df = (
        old_df[
            old_df["Day"] != (
                DAY_INDEX + 1
            )
        ]
    )

    updated_df = pd.concat(
        [
            old_df,
            current_result
        ],
        ignore_index=True
    )

else:

    updated_df = current_result

# ----------------------------------------
# Sort by Day
# ----------------------------------------

updated_df = (
    updated_df
    .sort_values("Day")
)

updated_df.to_csv(
    metrics_file,
    index=False
)

print(
    "Daily metrics saved."
)

In [ ]:
# 15

# ----------------------------------------
# UPDATE CHECKPOINT
# ----------------------------------------

next_mode = mode_number

next_day = day_number + 1

# ----------------------------------------
# If 31 days completed
# move to next mode
# ----------------------------------------

if next_day > 31:

    next_day = 1

    next_mode += 1

# ----------------------------------------
# SAVE UPDATED CHECKPOINT
# ----------------------------------------

with open(
    CHECKPOINT_FILE,
    "w"
) as f:

    f.write(
        f"MODE={next_mode}\n"
    )

    f.write(
        f"DAY={next_day}\n"
    )

print()

print(
    f"{MODE} | Day {day_number} completed."
)

print()

# ----------------------------------------
# Experiment Complete
# ----------------------------------------

if next_mode > 5:

    print(
        "ALL MODES COMPLETED."
    )

else:

    print(
        "Next Run Will Start From:"
    )

    print()

    print(
        f"Mode : mode_{next_mode}"
    )

    print(
        f"Day  : {next_day}"
    )

print()

print(
    "Results Directory:"
)

print(
    MODE_RESULTS_DIR
)